In [484]:
import pandas as pd
import math
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_error
import statistics

In [485]:
df = pd.read_csv("cabai merah_noEdit.csv", sep=";")
df.head()

,Komoditas (Rp),Semua Provinsi,Jawa Timur
0,Apr 2017 (I),58495,51150
1,Apr 2017 (II),59712,50700
2,Apr 2017 (III),51705,50850
3,Apr 2017 (IV),53092,51400
4,May 2017 (V),53954,56150


In [486]:
# min
# max
# z1, z2
# N
# R = Dmax + Z2 - (Dmin - Z1)
# mean = sum |Dt+1 - Dt|/N-1
# K = mean/2
# n = R/K

In [487]:
df_copy = df.copy()
# df_copy = df_copy.drop(index=df.index[0:209])
# df_copy.reset_index(inplace=True, drop=True)
df_copy

,Komoditas (Rp),Semua Provinsi,Jawa Timur
0,Apr 2017 (I),58495,51150
1,Apr 2017 (II),59712,50700
2,Apr 2017 (III),51705,50850
3,Apr 2017 (IV),53092,51400
4,May 2017 (V),53954,56150
...,...,...,...
302,Jan 2023 (IV),50138,45850
303,Feb 2023 (I),52962,54150
304,Feb 2023 (II),54819,58300
305,Feb 2023 (III),58716,59000


In [488]:
# print(np.var(df_copy['Jawa Timur']))
sd = statistics.pstdev(df_copy['Jawa Timur'].to_numpy())
meanS = statistics.mean(df_copy['Jawa Timur'].to_numpy())
print(sd/meanS*100)

55.43707512072425


In [489]:
Dt = df_copy['Jawa Timur']
Dmin = Dt.min()
Dmax = Dt.max()
N = Dt.count()
Z1 = 950
Z2 = 900

print(Dmin, Dmax, N)

11950 110100 307


In [490]:
R = Dmax+Z2 - (Dmin-Z1)
print(R)

100000


In [491]:
dif = []
df2 = df_copy

for i in range (N-1):
    dt = df['Jawa Timur'][i]
    dt2 = df['Jawa Timur'][i+1]
    ab = round(abs(dt2-dt),2)
    # dif.append(ab)
    df2.at[i, '|Dt+1 - Dt|'] = ab

# dif.append(0)
df2.at[N-1, '|Dt+1 - Dt|'] = 0

# df2 = df
# df2['|Dt+1 - Dt|'] = dif


df2.head()

,Komoditas (Rp),Semua Provinsi,Jawa Timur,|Dt+1 - Dt|
0,Apr 2017 (I),58495,51150,450.0
1,Apr 2017 (II),59712,50700,150.0
2,Apr 2017 (III),51705,50850,550.0
3,Apr 2017 (IV),53092,51400,4750.0
4,May 2017 (V),53954,56150,950.0


In [492]:
mean = (sum(df2['|Dt+1 - Dt|']))/(N-1)
print(mean)

4878.59477124183


In [493]:
K = mean/2
if(K<1):
    K = round(K,1)
else:
    K = math.ceil(K)

print(K)

2440


In [494]:
# math.ceil = roundup
n = math.ceil(R/K)
print(n)


41


In [495]:
# himpunan fuzzy
# batas atas, batas bawah, nilai tengah

In [496]:
himpFuzzy = pd.DataFrame()
ui = []
batasBawah = []
batasAtas = []
Ai = []
ui.append('u1')
Ai.append('A1')
batasBawah.append(Dmin-Z1)
batasAtas.append(batasBawah[0] + K)

for i in range(1, n):
    ui.append('u'+str(i+1))
    Ai.append('A'+str(i+1))
    batasBawah.append(batasAtas[i-1])
    batasAtas.append(batasBawah[i] + K)

himpFuzzy['ui'] = ui
himpFuzzy['Ai'] = Ai
himpFuzzy['batas bawah'] = batasBawah
himpFuzzy['batas atas'] = batasAtas

In [497]:
# himpFuzzy.head()
# himpFuzzy

In [498]:
# mi / nilai tengah
himpFuzzy['mi'] = (himpFuzzy['batas bawah'] + himpFuzzy['batas atas'])/2
himpFuzzy.head()

,ui,Ai,batas bawah,batas atas,mi
0,u1,A1,11000,13440,12220.0
1,u2,A2,13440,15880,14660.0
2,u3,A3,15880,18320,17100.0
3,u4,A4,18320,20760,19540.0
4,u5,A5,20760,23200,21980.0


In [499]:
print(himpFuzzy['batas bawah'][0], himpFuzzy['batas atas'][n-1])

11000 111040


In [500]:
data = df2['Jawa Timur']

In [501]:
# FLR dan FLRG
hasilFuzzy = []
for i in data:
    copyHimp = himpFuzzy
    cond1 = copyHimp['batas bawah'] <= i 
    cond2 = copyHimp['batas atas'] >= i 
    dfAi = copyHimp.where(cond1 & cond2) 
    dfAi = dfAi[~dfAi['ui'].isna()]
    fuzzyfikasi = dfAi['Ai'].iloc[0]
    hasilFuzzy.append(fuzzyfikasi)

df2['fuzzyfikasi'] = hasilFuzzy

In [502]:
df2.head()

,Komoditas (Rp),Semua Provinsi,Jawa Timur,|Dt+1 - Dt|,fuzzyfikasi
0,Apr 2017 (I),58495,51150,450.0,A17
1,Apr 2017 (II),59712,50700,150.0,A17
2,Apr 2017 (III),51705,50850,550.0,A17
3,Apr 2017 (IV),53092,51400,4750.0,A17
4,May 2017 (V),53954,56150,950.0,A19


In [503]:
nextState = []
for k in range(0, len(df2.index)-1):
    neSt = df2['fuzzyfikasi'].iloc[k+1]
    nextState.append(neSt)

# nextState.append('')

df2['next state'] = pd.Series(nextState)
df2.head()

,Komoditas (Rp),Semua Provinsi,Jawa Timur,|Dt+1 - Dt|,fuzzyfikasi,next state
0,Apr 2017 (I),58495,51150,450.0,A17,A17
1,Apr 2017 (II),59712,50700,150.0,A17,A17
2,Apr 2017 (III),51705,50850,550.0,A17,A17
3,Apr 2017 (IV),53092,51400,4750.0,A17,A19
4,May 2017 (V),53954,56150,950.0,A19,A19


In [504]:
# FLRG
dfFLRG = pd.DataFrame()
dfFLRG['Ai'] = himpFuzzy['Ai']
dfFLRG['FLRG'] = ''

# ambil df2 kecuali baris terakhir karena next statenya kosong
df3 = df2[:-1]

for f in range(len(dfFLRG.index)):
    searchAi = 'A'+str(f+1)
    
    new = df3[df3['fuzzyfikasi'].isin([searchAi])]
    group = new['next state']
    group = group.to_numpy()
    
    if len(group) > 0 :
        dfFLRG['FLRG'][f] = group

# dfFLRG.head()

In [505]:
# dfFLRG
dfFLRG.head()

,Ai,FLRG
0,A1,"[A1, A1, A1, A1, A2, A2]"
1,A2,"[A2, A2, A2, A1, A2, A3, A2, A2, A3, A2, A2, A..."
2,A3,"[A4, A9, A2, A4, A4, A4, A3, A2, A4, A2, A2, A..."
3,A4,"[A2, A2, A4, A5, A6, A4, A3, A5, A2, A4, A3, A..."
4,A5,"[A5, A6, A5, A8, A3, A5, A4, A7, A6, A4, A2, A..."


In [506]:
# hasil fuzzyfikasi
dfFLRG['defuzzyfikasi'] = ''

for g in range(len(dfFLRG.index)):
    ai = dfFLRG['Ai'].loc[g]
    flrg = dfFLRG['FLRG'].loc[g]
    hasil = 0

    if (len(flrg) == 0):
        fuzzyfikasi = himpFuzzy[himpFuzzy['Ai'] == ai]['mi']
        hasil = fuzzyfikasi.values[0]
    
    elif (len(flrg) == 1):
        fuzzyfikasi = himpFuzzy[himpFuzzy['Ai'] == flrg[0]]['mi']
        hasil = fuzzyfikasi.values[0]
        
    else:
        lenFlrg = len(flrg)
        for h in range(len(flrg)):
            mi = himpFuzzy[himpFuzzy['Ai'] == flrg[h]]['mi']
            bobot = (1/len(flrg)) * mi.values[0]
            hasil = hasil + bobot
            

    dfFLRG['defuzzyfikasi'][g] = hasil

dfFLRG.head()


,Ai,FLRG,defuzzyfikasi
0,A1,"[A1, A1, A1, A1, A2, A2]",13033.333333
1,A2,"[A2, A2, A2, A1, A2, A3, A2, A2, A3, A2, A2, A...",15451.351351
2,A3,"[A4, A9, A2, A4, A4, A4, A3, A2, A4, A2, A2, A...",18184.444444
3,A4,"[A2, A2, A4, A5, A6, A4, A3, A5, A2, A4, A3, A...",19052.0
4,A5,"[A5, A6, A5, A8, A3, A5, A4, A7, A6, A4, A2, A...",23606.666667


In [507]:
df2['yt'] = ''

for y in range (len(dfFLRG.index)):
    ai = dfFLRG['Ai'].loc[y]
    listAi = df2[df2['fuzzyfikasi'] == ai]
    yt = dfFLRG['defuzzyfikasi'][y]
    
    if len(listAi) > 0:
        idx = listAi.index
        for z in idx:
            df2.at[z+1,'yt'] = yt

In [508]:
df2.head()

,Komoditas (Rp),Semua Provinsi,Jawa Timur,|Dt+1 - Dt|,fuzzyfikasi,next state,yt
0,Apr 2017 (I),58495.0,51150.0,450.0,A17,A17,
1,Apr 2017 (II),59712.0,50700.0,150.0,A17,A17,53700.0
2,Apr 2017 (III),51705.0,50850.0,550.0,A17,A17,53700.0
3,Apr 2017 (IV),53092.0,51400.0,4750.0,A17,A19,53700.0
4,May 2017 (V),53954.0,56150.0,950.0,A19,A19,53700.0


In [509]:
# MAPE
y_actual = df2['Jawa Timur'][1:-1].to_numpy()
# print(y_actual)
y_predict = df2['yt'][1:-1].to_numpy()
# print(y_predict)
cekMape = mean_absolute_percentage_error(y_actual, y_predict)
print(cekMape*100)

13.094139237444727


In [510]:
# Root MSE squared=False
cekMSE = mean_squared_error(y_actual, y_predict, squared=False)
print(cekMSE)

6138.642342323997


In [511]:
print(n, Z1, Z2)

41 950 900


In [512]:
# # prediksi 5 periode berikutnya = 4 karena yg 1 udh diatas

# nextYt = 10
# lastIndex = len(df2.index)
# dfPredictNext = df2

# # print(len(df2.index))
# for n in range (nextYt-1):
#     # Add fuzzyfikai dr hasil predict
#     nilaiYt = dfPredictNext['yt'][lastIndex-1+n]
    
#     copyHimp = himpFuzzy
#     cond1 = copyHimp['batas bawah'] <= nilaiYt
#     cond2 = copyHimp['batas atas'] >= nilaiYt
#     dfAi = copyHimp.where(cond1 & cond2) 
#     dfAi = dfAi[~dfAi['ui'].isna()]
#     fuzzyfikasi = dfAi['Ai'].iloc[0]
#     dfPredictNext.at[lastIndex + n -1, 'fuzzyfikasi'] = fuzzyfikasi
#     # print(fuzzyfikasi)

#     # add predict
#     yt = dfFLRG['defuzzyfikasi'][dfFLRG['Ai'] == fuzzyfikasi]
#     dfPredictNext.at[lastIndex+n, 'yt'] = yt.values[0]
#     # print(yt.values[0])
    

In [513]:
# df2.drop(index=df2.index[-1],axis=0,inplace=True)
# dfPredictNext